# Stopping endless model/tool loops

Call limits count calls, not seconds. The scripted test double always requests a tool, so the infinite-agent behavior is reproducible. The real tools return normally but never report completion. A separate timeout is needed for a tool that never returns.

[Call limits](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-call-limit)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.faults import FailModelAfterToolLimit, ScriptedModel, looping_model, tool_request
from lab.tools import next_page, poll_job
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ModelFallbackMiddleware, ToolCallLimitMiddleware
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from model_config import run_config
from langchain.agents.middleware.model_call_limit import ModelCallLimitExceededError
from langchain.agents.middleware.tool_call_limit import ToolCallLimitExceededError


In [ ]:
# Execute
agent = create_agent(
    model=looping_model(job_id="DEMO-42"), tools=[poll_job],
    middleware=[ModelCallLimitMiddleware(run_limit=3), TraceMiddleware("04_model_limit", pretty=True)],
)
model_limit_result = agent.invoke({"messages": [("user", prompt("poll"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(model_limit_result["messages"], pretty=True)

### Switch model and tool after a page limit

The third `next_page` request is returned to the model as a tool error instead of ending the run. We then inject a failure only for the pagination model. `ModelFallbackMiddleware` switches to a named status model, which chooses `poll_job` instead. This is a deterministic routing demonstration, not a real model outage.

In [ ]:
# Execute
primary = looping_model("next_page", cursor=0, name="page-model")
fallback = ScriptedModel(
    name="status-model",
    responses=[
        tool_request("poll_job", job_id="DEMO-42"),
        AIMessage(content="The page limit was reached, so I switched to the job-status tool."),
    ],
)
agent = create_agent(
    model=primary, tools=[next_page, poll_job],
    middleware=[
        ModelFallbackMiddleware(fallback),
        ToolCallLimitMiddleware(tool_name="next_page", run_limit=2, exit_behavior="continue"),
        TraceMiddleware("04_tool_limit", pretty=True),
        FailModelAfterToolLimit(primary, tool_name="next_page"),
    ],
)
tool_limit_result = agent.invoke({"messages": [("user", prompt("pages"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(tool_limit_result["messages"], pretty=True)

### Run budget versus thread budget
The second invocation retains the thread counter. Change the thread ID for a fresh conversation budget.

In [ ]:
# Execute
agent = create_agent(
    model=looping_model(job_id="DEMO-42"), tools=[poll_job],
    middleware=[ModelCallLimitMiddleware(run_limit=2, thread_limit=3)],
    checkpointer=InMemorySaver(),
)
config = run_config("limited-thread")
first = agent.invoke({"messages": [("user", prompt("poll"))]}, config=config)

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(first["messages"], pretty=True)

In [ ]:
# Execute
second = agent.invoke({"messages": [("user", prompt("poll"))]}, config=config)

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(second["messages"], pretty=True)
print("Counter state:", {k: v for k, v in agent.get_state(config).values.items() if "count" in k})

### Global tool budgets and termination policies

Without `tool_name`, one budget covers every tool. Run budgets reset per invocation; thread budgets accumulate through the checkpointer. `continue` returns tool errors and allows another strategy, `error` raises immediately, and `end` terminates a single-tool scenario. Model limits support `end` or `error`.

In [ ]:
# Execute
global_agent = create_agent(model=looping_model(job_id="DEMO-42"), tools=[poll_job], middleware=[ToolCallLimitMiddleware(run_limit=1, exit_behavior="error")])
with expect_error(ToolCallLimitExceededError):
    global_agent.invoke({"messages": [("user", prompt("poll"))]}, config=run_config())

global_budget_stopped = True

In [ ]:
# Inspect output
print("Global tool budget raised:", global_budget_stopped)

In [ ]:
# Execute
ended_agent = create_agent(model=looping_model(job_id="DEMO-42"), tools=[poll_job], middleware=[ToolCallLimitMiddleware(tool_name="poll_job", run_limit=1, exit_behavior="end")])
ended_result = ended_agent.invoke({"messages": [("user", prompt("poll"))]}, config=run_config())


In [ ]:
# Inspect output
show_messages(ended_result["messages"], pretty=True)

In [ ]:
# Execute
model_error_agent = create_agent(model=looping_model(job_id="DEMO-42"), tools=[poll_job], middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="error")])
with expect_error(ModelCallLimitExceededError):
    model_error_agent.invoke({"messages": [("user", prompt("poll"))]}, config=run_config())
model_budget_stopped = True

In [ ]:
# Inspect output
print("Model budget raised:", model_budget_stopped)